# LangGraph 計算代理

以 Graph API 建立工具迴圈。模型節點決定是否呼叫工具。工具節點回傳結果。條件邊決定下一個節點。

學習目標：載入設定、建立代理、觀察結果，再測試 HTTP 服務。

官方範例使用 Anthropic。本實驗改用 ChatOpenAI，透過 LiteLLM 呼叫標準 Chat Completions API。保留計算工具與明確的節點流程。

## 1. 準備環境

從本資料夾執行 `uv sync --locked --group notebook`。
再執行 `uv run --group notebook jupyter lab tutorial.ipynb`。

In [ ]:
import sys
from pathlib import Path

assert sys.version_info[:2] == (3, 13), "請選擇 Python 3.13 核心。"
assert Path("agent.py").is_file(), "請從本專案資料夾啟動 Notebook。"
RUN_LIVE = False
print("Python 版本：", sys.version.split()[0])

## 2. 檢查設定

設定來自環境變數或 `.env`。此步驟不輸出金鑰。
離線學習不需要有效設定。真實推論需要模型別名。

In [ ]:
from settings import ConfigurationError, Settings

try:
    settings = Settings.load()
    print("模型：", settings.model)
    print("時間限制：", settings.timeout)
except ConfigurationError as exc:
    settings = None
    print(str(exc))

## 3. 先測試工具

工具是具有型別與說明的 Python 函式。以下呼叫不需要 LLM。

In [ ]:
from agent import add, divide, multiply

subtotal = add.invoke({"a": 12, "b": 8})
product = multiply.invoke({"a": subtotal, "b": 3})
print(divide.invoke({"a": product, "b": 5}))

## 4. 檢查流程

`StateGraph` 儲存訊息。`model` 節點呼叫 LLM。
若回答包含工具呼叫，條件邊會前往 `tools`。
工具回傳後，流程再回到 `model`。

In [ ]:
from agent import build_agent
from model import create_model

if settings is not None:
    graph = build_agent(create_model(settings))
    print(graph.get_graph().draw_mermaid())
else:
    print("設定完成後，可在此檢查圖形結構。")

## 5. 執行真實任務

確認 `.env` 的端點、金鑰與模型名稱。
將第一個程式區塊中的 `RUN_LIVE` 改為 `True`。
重新執行設定步驟，再執行下列區塊。這會使用模型額度。

In [ ]:
from agent import run_agent

result = None
if RUN_LIVE:
    result = await run_agent("請使用工具計算 (12 + 8) × 3，再除以 5。")
    print(result["output"])
else:
    print("已略過真實推論。將 RUN_LIVE 改為 True 可執行。")

## 6. 觀察結果

檢查工具結果或代理輸出。每次執行都是獨立任務。

In [ ]:
if result is not None:
    print(result.get("tool_results", []))
else:
    print("尚未執行真實任務。")

## 7. 測試 Docker 服務

在終端機執行 `docker compose up --build -d --wait`。
開啟 `http://localhost:8001/docs`。
先呼叫 `/health`，再呼叫 `/invoke`。
完成後執行 `docker compose down`。
若 LiteLLM 只監聽 Linux 主機的 `127.0.0.1`，請改用下列命令：

```bash
docker compose -f compose.yaml -f docker/compose.host.yaml up --build -d --wait
```

停止時使用 `docker compose -f compose.yaml -f docker/compose.host.yaml down`。


In [ ]:
RUN_HTTP = False
if RUN_HTTP:
    import httpx

    async with httpx.AsyncClient(timeout=180) as client:
        response = await client.post(
            "http://localhost:8001/invoke",
            json={"prompt": "請使用工具計算 (12 + 8) × 3，再除以 5。"},
        )
        response.raise_for_status()
        print(response.json())

## 8. 延伸練習

新增減法工具。將工具加入 TOOLS。再觀察流程是否呼叫新工具。

## 官方來源

- [LangGraph 官方 Quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart)
- [ChatOpenAI 官方整合文件](https://docs.langchain.com/oss/python/integrations/chat/openai)